# Graph Neural Networks for Crystals
*AI School 26 · Aachen*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/paulopires0/aachen-school/blob/main/workshop_starter.ipynb)

In [ ]:
import os, subprocess, sys

REPO = "https://github.com/paulopires0/aachen-school.git"

if "google.colab" in sys.modules:
    if os.path.basename(os.getcwd()) != "aachen-school":
        if not os.path.isdir("aachen-school"):
            subprocess.run(["git", "clone", "-q", REPO], check=True)
        os.chdir("aachen-school")
    try:
        import pymatgen.core          # numpy / torch / matplotlib / sklearn: already on Colab
    except ImportError:
        print("installing pymatgen, about a minute ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pymatgen"], check=True)
        import pymatgen.core          # if THIS line fails: Runtime > Restart session, run again

print("working folder:", os.getcwd())

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import workshop as wk

np.random.seed(0)
torch.manual_seed(0)

data = wk.load_data()        

print(data.formulas[0], "| measured theta =", data.theta[0], "K")
print(data.structures[0])

---
# Part 1 · MLP
## 1a. In NumPy

In [ ]:
x = np.array([0.8, -1.2, 0.4, 1.5])          # inputs
W = np.random.default_rng(0).normal(size=(5, 4))   # weights
b = np.random.default_rng(1).normal(size=5)        # bias 

def relu(z):
    # TODO: keep the positive numbers, set the negative ones to zero
    return None

def neuron(x, W, b):
    # TODO: multiply (W @ x), add the bias, and activate it with relu
    return None

h_np = neuron(x, W, b)
print("numpy:", h_np.round(3))

## 1b. In PyTorch

In [ ]:
layer = nn.Linear(4, 5).double()              #nn.Linear(4, 5) is the W @ x + b above, with the weights stored inside it.
with torch.no_grad():                         # put our W and b into the layer as torch tensors
    layer.weight.copy_(torch.tensor(W))
    layer.bias.copy_(torch.tensor(b))

xt = torch.tensor(x)  # We need to convert x into a torch tensor

# TODO: the same neuron in torch: run the layer on xt, then torch.relu
h_torch = None

print("numpy:", h_np.round(3))
print("torch:", h_torch.detach().numpy().round(3))
print("same :", np.allclose(h_np, h_torch.detach().numpy()))

## 1c. A MLP

In [ ]:
rng = np.random.default_rng(0)
SIZES = [4, 5, 4, 3]
PARAMS = [(rng.normal(size=(o, i)), rng.normal(size=o)) for i, o in zip(SIZES[:-1], SIZES[1:])]

def mlp(v):
    # TODO: Cycle through all but the last layer, applying neuron() to each layer
    #       last layer is linear (an activation function kills the signal), then create a pooling 
    return None           

print("prediction:            ", round(mlp(x), 4))
print("same inputs, new order:", round(mlp(x[[2, 0, 3, 1]]), 4))

## 1d. Example of torch MLP

In [ ]:
def mlp_torch(v):
    layers = []
    for k, (i, o) in enumerate(zip(SIZES[:-1], SIZES[1:])):
        layer = nn.Linear(i, o).double()
        W_, b_ = PARAMS[k]         

        with torch.no_grad():
            layer.weight.copy_(torch.tensor(W_))
            layer.bias.copy_(torch.tensor(b_))

        layers.append(layer)

        if k < len(SIZES) - 2:  
            layers.append(nn.ReLU())
            
    return nn.Sequential(*layers)(torch.tensor(v))

---
# Part 2 · Crystal graphs


## 2a. Atomic representations

In [ ]:
from pymatgen.core import Element

def features(struct):
    # TODO: Create a feature vector for each site in the structure
    #       Use pymatgen.core.Element to get the atomic number, electronegativity etc... of each site
    #       Or do a one-hot encoding of the element type
    #       Every atom must get the same number of values. 
    return None

print(data.formulas[0])
print(features(data.structures[0]).round(2))

## 2b. Creating the graph
`struct.get_neighbor_list(r)` gives every pair closer than `r`, periodic copies included, as four arrays:
centre atom, neighbour, which copy of the cell the neighbour sits in, and the distance. Use
`wk.keep_nearest(...)` to trim that to the k nearest neighbours of each atom.

In [ ]:
def graph(struct):
    # TODO: (1) struct.get_neighbor_list(r=5.0)
    #       (2) wk.keep_nearest(centre, neighbour, image, distance, k=12)
    #       (3) return centre, neighbour, distance
    return None

c, n, d = graph(data.structures[0])
print(graph(data.structures[0]))
print(len(data.structures[0]), "atoms ->", len(c), "edges")
print("shortest bond", d.min().round(2), "A   longest kept", d.max().round(2), "A")

---
# Part 3 · Message passing, twice

Lets use the MLP from the first step as the function that modulates the message:

$$h_i \;\leftarrow\; h_i + \sum_{j \in \text{neighbours}(i)} MLP(h_i|d_{ij}|h_j)$$

(a|b) referes to the concatenation of the vectors, use np.concatenate((a,b), axis=1)

## 3a. In NumPy

In [ ]:
def MLP_message():
    # TODO: Recreate the MLP from above, however note that:
    #       (1) the input is now twice the size of the feature you made + 1 of the distance
    #       (2) the output is now the same size as the feature you made so sum is possible
    return None

MSG = MLP_message()      # build it once here, so 3b can reuse exactly these weights

def message_np(h, c, n, d):
    v = np.concatenate([h[c], h[n], d[:, None]], axis=1)
    # TODO: (1) compute the messages from the neighbours to the centre atom:
    #           feed h[c], h[n] and d through MLP. Careful, h[c] is a stack of edges,
    #           so a layer is  v @ W.T + b  and not  W @ v  as in the single-vector case.
    #       (2) aggregate the messages for each centre atom with a sum:
    #           np.add.at(agg, c, msg) adds every message onto its centre atom
    agg = None
    return h + agg

def pool_np(h):
    # TODO: average/max/sum over the atoms
    return None

h = features(data.structures[0])
for _ in range(3):
    h = message_np(h, c, n, d)
print("crystal vector:", pool_np(h).round(3))
print("prediction with random weights:", round(mlp(pool_np(h)), 4))

## 3b. The same two steps in PyTorch

* `torch.zeros_like(h).index_add_(0, c, msg)` is the torch version of `np.add.at`;

In [ ]:
def MLP_message_torch():
    # TODO: Recreate the MLP from above using torch and use the same weights as in the numpy version
    #       nn.Linear(in, out) for every layer, then copy the numbers out of MSG into them
    #       with weight.copy_ / bias.copy_ , exactly as in 1b
    return None

MSG_TORCH = MLP_message_torch()

def message_torch(h, c, n, d):
    # TODO: (1) compute the messages using the MLP
    #       (2) aggregate the messages for each centre atom (index_add_ is the sum)
    #       use torch.cat for concatenation
    msg = None
    return h + torch.zeros_like(h).index_add_(0, c, msg)

def pool_torch(h):
    return h.mean(dim=0)

ht = torch.tensor(features(data.structures[0]), dtype=torch.float32)
ct, nt, dt = torch.tensor(c), torch.tensor(n), torch.tensor(d, dtype=torch.float32)
for _ in range(3):
    ht = message_torch(ht, ct, nt, dt)
pooled = pool_torch(ht)

print("numpy:", pool_np(h).round(3))
print("torch:", pooled.detach().numpy().round(3))

---
# Part 4 · Build the GNN and train it

### The target
Theta is an invented property, but it follows a fixed formula:

$$\omega = 800\,\mathrm{K}\sqrt{\tfrac{20\,\mathrm{u}}{\langle m\rangle}}\,\tfrac{2.5\,\text{A}}{\langle d\rangle}
\qquad \lambda = 0.25 + 0.9\,\langle|\chi_i-\chi_j|\rangle \qquad \Theta = \omega\,e^{-1/\lambda}$$


In [ ]:
from sklearn.model_selection import train_test_split

# TODO: visualize the distribution of theta values in the dataset (data.theta), and how many are above 250 K?

def split(theta, seed=0):
    # TODO: split the data into training, validation and test sets. 
    # Use train_test_split() twice, with random_state=0 for reproducibility
    return None
i_tr, i_va, i_te = split(data.theta)


### Turn every crystal into tensors

In [ ]:
graphs = []
for st_ in data.structures:
    c_, n_, d_ = graph(st_)
    graphs.append((torch.tensor(features(st_), dtype=torch.float32),
                   torch.tensor(c_), torch.tensor(n_), torch.tensor(d_, dtype=torch.float32)))
print(len(graphs), "graphs ready")

### 4a. The model

In [ ]:
def pool_batch(h, b, n_graphs):
    # several crystals are stacked now, so the plain mean of pool_torch is not enough:
    # b says which crystal each atom belongs to, so average the atoms of each one separately
    total = torch.zeros(n_graphs, h.shape[1]).index_add_(0, b, h)   # sum the atoms of each crystal
    count = torch.bincount(b, minlength=n_graphs)[:, None]    # atoms per crystal
    return total / count


class MyGNN(nn.Module):
    def __init__(self, dim=64, layers=3, out=1):
        super().__init__()
        # TODO: Create self.embed, a Linear layer to embed the atom features
        # TODO: Create self.phi, a list of MLPs for the message passing, one for each layer
        #       Use nn.ModuleList() to store them, and nn.Sequential() to create each MLP
        # TODO: Create self.head, a Linear layer to output the final prediction

    def forward(self, x, c, n, d, b, n_graphs):
        h = self.embed(x)
        for phi in self.phi:
            msg = phi(torch.cat([h[c], h[n], d[:, None]], dim=1))
            # TODO: add the messages to their centre atoms, as in message_torch
            #       (divide by 12 so that the size does not blow up: h + ... / 12)
            h = None
        # TODO: pool the atoms into one vector per crystal, with pool_batch
        pooled = None
        return self.head(pooled)

model = MyGNN()
print(sum(p.numel() for p in model.parameters()), "weights to learn")

### 4b. The loss

In [ ]:
def loss(pred, target):
    # TODO: Define a loss function that computes the average of |pred - target|
    return None

history = wk.train(model, loss, graphs, data.theta, i_tr, i_va, epochs=20, lr=1e-3)
# TODO: plot the training and validation loss curves stored in history

### 4c. Is it any good?

In [ ]:
pred_va = wk.predict(model, graphs, i_va)

# TODO: compute the validation MAE of your GNN, and compare it to the median baseline

# TODO: visualize the distribution of errors per theta value

### 4d. Play
Change one thing, retrain, and watch the curves:

* `epochs=40` — does the validation loss keep falling?
* `lr=1e-2`, then `lr=1e-4` — too big jumps around, too small barely moves
* `MyGNN(layers=1)` or `MyGNN(layers=5)` — how far does an atom need to see?
* `MyGNN(dim=16)` — a smaller model; how much does it lose?

In [ ]:
# TODO: Load the MyGNN again but with a smaller number of parameters (e.g. dim=16, layers=1) and train it again

# TODO: plot the training and validation loss curves of the small model against the big one

---
# Part 5 · Playing with GNNs


### 5a. A loss for high Theta

In [ ]:
def weighted_loss(pred, target, beta=8.0, theta0=250):   # theta0 in K, the threshold we care about
    # TODO: Create a weighted loss to target high values
    return None

# TODO: Train a model with the new loss, plot the distrubition of error per theta value

### 5b. Physical informed GNNs

In [ ]:
import torch.nn.functional as F

def compute_theta (lam, omega):
    # TODO: compute theta from lambda and omega 
    return None

def physics_loss(out, target):
    # TODO: Make a loss function that computes the error of the predicted theta from the predicted lambda and omega
    return None

# TODO: Train a model with the new loss, plot the distrubition of error per theta value
#Note that the model will initialy predict random values of lambda
#A lambda close to 0 makes exp(-1/lambda) explode, so keep it away from 0 with
#0.25 + F.softplus(...) 
#Omega spans 155..11881 K, too wide to predict directly: predict its log

### 5c. Parity

In [ ]:
# TODO: Make a parity plot of the predicted vs true values of theta for the three models 

# TODO: Make the parity plot for lamdba and omega as well

# TODO: Compare the error of the three models

### 5e. Classification

In [ ]:
THRESHOLD = 250        # K - try other values

# TODO: Compute True/False Positives and Negatives for one of the models

# TODO: Make a confusion matrix, and also plot precision vs recall